In [7]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

CUDA available: True
GPU: Tesla T4


# 04D — Portable local / Colab setup and experiments

Default: **non-training smoke test**. Locally, use the current checkout and a pinned Python environment; set `EDGEAI_DATA_ROOT` before importing tools to use an external EdgeAI storage tree. Colab mounts Drive, clones or fast-forward pulls the GitHub repository, and installs its pinned runtime requirements. Code executes in the connected kernel's filesystem.

Upload the complete staged `EdgeAI` tree to `/content/drive/MyDrive/EdgeAI` and commit/push local code fixes before the first Colab run. Never run 00–03B to repair a missing cache. This notebook does not push code or upload data.


In [8]:
from pathlib import Path, PureWindowsPath
import os, subprocess, sys
try:
    from google.colab import drive
except ImportError:
    drive = None
IN_COLAB = drive is not None

REPO_URL = "https://github.com/ancaranovik/Mosquito_Wingbeat_v1.git"
if IN_COLAB:
    drive.mount("/content/drive")
    assert REPO_URL.startswith("https://github.com/") and "@" not in REPO_URL
    repo_hint = os.environ.get("EDGEAI_REPO_ROOT", "/content/edge-ai")
    if PureWindowsPath(repo_hint).drive:
        raise ValueError("Colab needs a runtime checkout, not a Windows repository path")
    repo = Path(repo_hint).resolve()
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    else:
        origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=repo, text=True).strip()
        assert origin == REPO_URL, "Existing checkout has a different origin"
        assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip(), "Preserve uncommitted runtime changes before pulling"
        subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
else:
    hint = Path(os.environ.get("EDGEAI_REPO_ROOT", Path.cwd())).resolve()
    repo = next((p for p in [hint, *hint.parents] if (p / "tools/project_paths.py").is_file()), None)
    if repo is None:
        raise RuntimeError("Set EDGEAI_REPO_ROOT to your local checkout")
os.environ["EDGEAI_REPO_ROOT"] = str(repo)
os.chdir(repo)
sys.path.insert(0, str(repo / "tools"))
sys.dont_write_bytecode = True
print("Mode:", "COLAB" if IN_COLAB else "LOCAL")
print("Checkout:", repo)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mode: COLAB
Checkout: /content/edge-ai


In Colab, install the pinned cache-consumer and notebook dependencies. Locally, use the already configured environment without installing packages. Verification and smoke testing use fresh child Python processes. Configure private GitHub authentication interactively; never embed credentials here.


In [9]:
from colab_bootstrap import configure
paths = configure(install=IN_COLAB)


Repository: /content/edge-ai
Data: /content/drive/MyDrive/EdgeAI
Immutable baseline: /content/drive/MyDrive/EdgeAI/fixed/baseline/stage04
Future results: /content/drive/MyDrive/EdgeAI/experiments/<experiment_id>


Verify all frozen identities and eight saved baseline runs. This reads saved predictions and hashes; it does not evaluate a trained model on TEST again or regenerate any data.

In [10]:
subprocess.run([sys.executable, "-B", "tools/verify_portability.py"], cwd=repo, check=True)


CompletedProcess(args=['/usr/bin/python3', '-B', 'tools/verify_portability.py'], returncode=0)

Colab requires CUDA and reports the GPU name before eight synthetic forward checks. Local mode uses CPU. Neither mode constructs an optimizer or starts training.


In [11]:
device = "cuda" if IN_COLAB else "cpu"
subprocess.run([sys.executable, "-B", "tools/experiment_runner.py", "smoke", "--device", device], cwd=repo, check=True)
print("Future outputs:", paths.EXPERIMENT_ROOT / "<experiment_id>")


Future outputs: /content/drive/MyDrive/EdgeAI/experiments/<experiment_id>


## Future runs (disabled)

Copy `configs/experiments/baseline_reuse.json` to a new config locally, choose a never-used `exp_...` ID, commit and push, then pull here. The current runner deliberately supports only the accepted CUDA protocol and four existing architectures. New scientific options need separately reviewed implementations. Changes to waveform/segmentation/features belong in a new versioned `derived/` location. Existing experiments cannot be resumed or overwritten implicitly.

In [12]:
RUN_TRAINING = False
EXPERIMENT_CONFIG = "configs/experiments/baseline_reuse.json"
if RUN_TRAINING:
    subprocess.run([sys.executable, "-B", "tools/experiment_runner.py", "train", "--config", EXPERIMENT_CONFIG], cwd=repo, check=True)
else:
    print("Training disabled. Infrastructure smoke test only.")


Training disabled. Infrastructure smoke test only.
